# 04 - Optimal Commute Routing with Multi-Candidate Stations

This notebook calculates optimal door-to-door commute times using OpenTripPlanner for transit routing between multiple candidate stations for each employee and the office, selecting the combination that minimizes total commute time.

**How it works:**
- For each employee, we have 5 candidate stations from the previous notebook
- For the office, we have 5 candidate stations from the previous notebook
- We evaluate all 25 combinations (5 employee stations × 5 office stations)
- For each combination, we calculate: walk_home_to_station + transit_time + walk_station_to_office
- We select the combination with the minimal total commute time

**Formula:** walk_home_to_station + transit_time + walk_station_to_office

In [ ]:
# Import the libraries we need for routing
import requests
import pandas as pd
import time
from dotenv import load_dotenv
import os

In [ ]:
# Load environment variables and configure OpenTripPlanner server
load_dotenv()

# OpenTripPlanner server configuration
OTP_BASE_URL = "http://localhost:8080/otp/routers/default"

# Load the office candidate stations with walking times from the previous notebook
office_candidates_df = pd.read_csv('data/jj_office_candidate_stations_with_walking.csv')
print(f"Loaded {len(office_candidates_df)} office candidate stations")
print(office_candidates_df[['station_name', 'candidate_rank', 'walking_time_min']])

# Load the employee candidate stations with walking times from the previous notebook
employee_candidates_df = pd.read_csv('data/employee_candidate_stations_with_walking.csv')
print(f"\nLoaded {len(employee_candidates_df)} employee candidate stations")
print(f"Employees with candidates: {employee_candidates_df['employee_id'].nunique()}")

# Display sample data to verify
print("\nSample employee candidates:")
print(employee_candidates_df[['employee_id', 'station_name', 'candidate_rank', 'walking_time_min']].head(10))

In [ ]:
# Load the original employee data to store our final routing results
employee_df = pd.read_csv('data/synthetic_employees_geocoded.csv')
print(f"Loaded {len(employee_df)} employees")

In [ ]:
# Function to get transit route between two stations using OpenTripPlanner API
def get_transit_route(from_lat, from_lon, to_lat, to_lon, date, time_str):
    """Get transit route using OpenTripPlanner API"""
    url = f"{OTP_BASE_URL}/plan"
    
    params = {
        'fromPlace': f"{from_lat},{from_lon}",
        'toPlace': f"{to_lat},{to_lon}",
        'mode': 'TRANSIT,WALK',
        'date': date,
        'time': time_str
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'error' in data and data['error']['id'] == 406:
            # No transit times available - this is okay, return walking only
            return data
        
        return data
        
    except Exception as e:
        print(f"Error fetching route: {e}")
        return None

In [ ]:
# Test the routing function with a sample employee candidate and office candidate
sample_employee_candidate = employee_candidates_df.iloc[0]
sample_office_candidate = office_candidates_df.iloc[0]

print(f"Testing routing with sample stations:")
print(f"Employee: {sample_employee_candidate['employee_id']}")
print(f"Employee station: {sample_employee_candidate['station_name']} (rank {sample_employee_candidate['candidate_rank']})")
print(f"Office station: {sample_office_candidate['station_name']} (rank {sample_office_candidate['candidate_rank']})")

from_lat = sample_employee_candidate['station_latitude']
from_lon = sample_employee_candidate['station_longitude']
to_lat = sample_office_candidate['station_latitude']
to_lon = sample_office_candidate['station_longitude']

print(f"From: ({from_lat}, {from_lon})")
print(f"To: ({to_lat}, {to_lon})")

# Set date for routing (within GTFS range - our GTFS data has April-December 2026)
route_date = "2026-07-08"
route_time = "08:00"

route_data = get_transit_route(from_lat, from_lon, to_lat, to_lon, route_date, route_time)

if route_data and 'plan' in route_data:
    if route_data['plan']['itineraries']:
        itinerary = route_data['plan']['itineraries'][0]
        print(f"Transit time: {itinerary['transitTime']}s ({itinerary['transitTime']/60:.1f} min)")
        print(f"Walking time: {itinerary['walkTime']}s ({itinerary['walkTime']/60:.1f} min)")
        print(f"Total time: {itinerary['duration']}s ({itinerary['duration']/60:.1f} min)")
        
        # Calculate total commute using our walking data from the previous notebook
        employee_walk = sample_employee_candidate['walking_time_min']
        office_walk = sample_office_candidate['walking_time_min']
        total_using_our_data = employee_walk + (itinerary['transitTime']/60) + office_walk
        print(f"Using our walking data: {employee_walk:.1f} + {itinerary['transitTime']/60:.1f} + {office_walk:.1f} = {total_using_our_data:.1f} min")
    else:
        print("No itineraries found")
else:
    print("Error in routing")

In [ ]:
# Set up the multi-candidate routing process
print("Starting multi-candidate optimal routing...")
print(f"Employees: {employee_df['employee_id'].nunique()}")
print(f"Employee candidates per employee: 5")
print(f"Office candidates: {len(office_candidates_df)}")
print(f"Total combinations to evaluate: {employee_df['employee_id'].nunique() * 5 * 5}")

print(f"\nUsing date: {route_date}, time: {route_time}")
print(f"Formula: walk_home_to_station + transit_time + walk_station_to_office")

In [ ]:
# Main routing loop - evaluate all station combinations for each employee


employees_processed = 0
combinations_evaluated = 0
cache_file = 'data/routing_cache.csv'

# Load cache if exists to avoid recalculating the same routes
if os.path.exists(cache_file):
    print("Loading routing cache...")
    routing_cache = pd.read_csv(cache_file)
    print(f"Loaded {len(routing_cache)} cached routes")
else:
    routing_cache = pd.DataFrame(columns=['cache_key', 'employee_station_id', 'office_station_id', 'transit_time_min', 'route_found'])
    print("No cache found, starting fresh")

print("\nProcessing employees...")

# Process each employee individually
for employee_id in employee_df['employee_id']:
    print(f"\nProcessing {employee_id} ({employees_processed + 1}/{len(employee_df)})...")
    
    # Get this employee's 5 candidate stations
    emp_candidates = employee_candidates_df[employee_candidates_df['employee_id'] == employee_id]
    
    best_total_time = float('inf')
    best_route = None
    
    # Evaluate all combinations of employee station × office station (5 × 5 = 25 combinations)
    for _, emp_candidate in emp_candidates.iterrows():
        for _, office_candidate in office_candidates_df.iterrows():
            
            # Check if this combination is already cached
            cache_key = f"{emp_candidate['station_id']}_{office_candidate['station_id']}"
            if len(routing_cache) > 0 and cache_key in routing_cache['cache_key'].values:
                # Use cached transit time to save API calls
                cached_route = routing_cache[routing_cache['cache_key'] == cache_key].iloc[0]
                transit_time = cached_route['transit_time_min']
                route_found = cached_route['route_found']
            else:
                # Calculate transit time via OpenTripPlanner
                route_data = get_transit_route(
                    emp_candidate['station_latitude'], emp_candidate['station_longitude'],
                    office_candidate['station_latitude'], office_candidate['station_longitude'],
                    route_date, route_time
                )
                
                if route_data and 'plan' in route_data and route_data['plan']['itineraries']:
                    itinerary = route_data['plan']['itineraries'][0]
                    transit_time = itinerary['transitTime'] / 60
                    route_found = True
                else:
                    transit_time = None
                    route_found = False
                
                # Add to cache
                new_cache_entry = pd.DataFrame([{
                    'cache_key': cache_key,
                    'employee_station_id': emp_candidate['station_id'],
                    'office_station_id': office_candidate['station_id'],
                    'transit_time_min': transit_time,
                    'route_found': route_found
                }])
                routing_cache = pd.concat([routing_cache, new_cache_entry], ignore_index=True)
            
            # Calculate total commute time for this combination
            if route_found and transit_time is not None:
                total_time = emp_candidate['walking_time_min'] + transit_time + office_candidate['walking_time_min']
                
                # Track the best route for this employee
                if total_time < best_total_time:
                    best_total_time = total_time
                    best_route = {
                        'employee_station': emp_candidate['station_name'],
                        'employee_station_id': emp_candidate['station_id'],
                        'employee_station_rank': emp_candidate['candidate_rank'],
                        'office_station': office_candidate['station_name'],
                        'office_station_id': office_candidate['station_id'],
                        'office_station_rank': office_candidate['candidate_rank'],
                        'walk_home_to_station_min': emp_candidate['walking_time_min'],
                        'transit_time_min': transit_time,
                        'walk_station_to_office_min': office_candidate['walking_time_min'],
                        'total_commute_min': total_time,
                        'otp_route_found': True
                    }
            
            combinations_evaluated += 1
    
    # Store the best route for this employee
    if best_route:
        employee_idx = employee_df[employee_df['employee_id'] == employee_id].index[0]
        for key, value in best_route.items():
            employee_df.at[employee_idx, key] = value
    else:
        # No valid route found for this employee
        employee_idx = employee_df[employee_df['employee_id'] == employee_id].index[0]
        employee_df.at[employee_idx, 'otp_route_found'] = False
    
    employees_processed += 1
    
    # Save cache every 10 employees to avoid losing progress
    if employees_processed % 10 == 0:
        routing_cache.to_csv(cache_file, index=False)
        print(f"\nProgress: {employees_processed}/{len(employee_df)} employees processed")
        print(f"Combinations evaluated: {combinations_evaluated}")
        print(f"Cache size: {len(routing_cache)} routes")

# Save final cache
routing_cache.to_csv(cache_file, index=False)
print(f"\n✅ Routing complete! Processed {employees_processed} employees, evaluated {combinations_evaluated} combinations.")

In [ ]:
# Summary of routing results
print(f"\n=== Routing Summary ===")
print(f"Total employees processed: {employees_processed}")
print(f"Total combinations evaluated: {combinations_evaluated}")
print(f"Employees with valid routes: {employee_df['otp_route_found'].sum()}")
print(f"Employees without routes: {(~employee_df['otp_route_found']).sum()}")

print(f"\nAverage commute time for employees with routes: {employee_df[employee_df['otp_route_found']]['total_commute_min'].mean():.1f} minutes")
print(f"Average car time: {employee_df['car_total_time_min'].mean():.1f} minutes")

# Display sample of optimal routes
print(f"\nSample optimal routes:")
print(employee_df[['employee_id', 'optimal_employee_station', 'optimal_employee_station_rank', 
                   'optimal_office_station', 'optimal_office_station_rank', 
                   'walk_home_to_station_min', 'transit_time_min', 'walk_station_to_office_min', 
                   'total_commute_min']].head(10))

In [ ]:
# Save the final results with optimal commute routes
employee_df.to_csv('data/synthetic_employees_with_optimal_commutes.csv', index=False)
print("Saved optimal commute results to data/synthetic_employees_with_optimal_commutes.csv")